This notebook is for generating "silver label" examples using the trained span identification and technique classification models in order to train a lighter weight model. The raw news article data pre-adding silver labels is from the English-only subset of the Common Crawl News dataset. Once run through the existing models to get "silver labels," we use these examples to train a xx model to be used in our Chrome extension.

In [61]:
import pandas as pd
import numpy as np
from pathlib import Path
from datasets import load_dataset
import os
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification
import json
from tqdm.auto import tqdm
import spacy
from textblob import TextBlob
import re
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, classification_report, precision_recall_curve
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import chi2

In [2]:
#Identify base directory to ensure portability
BASE_DIR = Path.cwd().resolve().parent
MODELS_DIR = BASE_DIR / "models"
interim_dir = BASE_DIR / "data" / "interim"
interim_dir.mkdir(parents=True, exist_ok=True)
output_file = interim_dir / "news_with_labels.csv"
second_output_file = Path("../data/processed/news_with_features.csv")
DATA_PATH = BASE_DIR / "data" / "processed" / "semeval_tc_cleaned.csv"

SI_DIR = MODELS_DIR / "semeval_roberta_scanner"
SI_SPEC_DIR = MODELS_DIR / "semeval_roberta_scanner_specialist"
TC_DIR = MODELS_DIR / "semeval_roberta_classifier"

SI_MODEL_PATH = f"{os.fspath(SI_DIR.absolute())}"
SI_SPEC_PATH = f"{os.fspath(SI_SPEC_DIR.absolute())}"
TC_MODEL_PATH = f"{os.fspath(TC_DIR.absolute())}"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [3]:
#Run training notebooks if models are missing
REQUIRED_FILES = ["config.json", "model.safetensors"]

def model_exists(path):
    path = Path(path)
    has_weights = any(path.glob("*.bin")) or any(path.glob("*.safetensors"))
    return has_weights

if not model_exists(SI_MODEL_PATH):
    print("SI Model missing. Running training notebook...")
    %run 4.1-fp-semeval-si-modeling.ipynb
if not model_exists(TC_MODEL_PATH):
    print("TC Model missing. Running training notebook...")
    %run 4.2-fp-semeval-tc-modeling.ipynb

In [4]:
#Load Base SI Model (RoBERTa token-classifier for span detection)
print(f"Loading Base SI Model from: {SI_MODEL_PATH}...")
si_tokenizer = AutoTokenizer.from_pretrained(SI_MODEL_PATH)
si_model = AutoModelForTokenClassification.from_pretrained(SI_MODEL_PATH, local_files_only=True).to(device)
si_model.eval()

#Load Specialist SI Model
print(f"Loading Specialist SI Model from: {SI_SPEC_PATH}...")
si_spec_model = AutoModelForTokenClassification.from_pretrained(SI_SPEC_PATH, local_files_only=True).to(device)
si_spec_model.eval()

Loading Base SI Model from: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_scanner...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading Specialist SI Model from: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_scanner_specialist...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForTokenClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (L

In [5]:
#Load TC Model (Technique Classification)
tc_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
tc_model = AutoModelForSequenceClassification.from_pretrained(TC_MODEL_PATH).to(device)
tc_model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50267, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.2, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [6]:
#Set thresholds for each technique
OPTIMIZED_THRESHOLDS = {
    'Appeal_to_Authority': 0.60,
    'Appeal_to_fear-prejudice': 0.50,
    'Bandwagon_Reductio_ad_hitlerum': 0.10,
    'Black-and-White_Fallacy': 0.20,
    'Causal_Oversimplification': 0.20,
    'Doubt': 0.35,
    'Exaggeration_Minimisation': 0.40,
    'Flag-Waving': 0.45,
    'Loaded_Language': 0.40,
    'Name_Calling_Labeling': 0.55,
    'Repetition': 0.40,
    'Slogans': 0.30,
    'Thought-terminating_Cliches': 0.15,
    'Whataboutism_Straw_Men_Red_Herring': 0.15
}

In [7]:
def run_pipeline_batched(texts):
    """
    Processes a list of texts through the SI -> Cascade -> TC pipeline.
    """
    #1. SI Tokenization (Batch)
    inputs = si_tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True
    ).to(device)

    offsets_batch = inputs.pop("offset_mapping")

    with torch.no_grad():
        #2. SI Model Inference (Batch)
        base_outputs = si_model(**inputs)
        base_preds_batch = torch.argmax(base_outputs.logits, dim=-1)

        # 3. Specialist Model Inference (Batch)
        spec_outputs = si_spec_model(**inputs)
        spec_probs_batch = F.softmax(spec_outputs.logits, dim=-1)
        propaganda_prob_batch = spec_probs_batch[:, :, 1]

    #4. Apply Cascade Logic & Extract Spans for each item in batch
    batch_final_results = []

    for i in range(len(texts)):
        text = texts[i]
        base_preds = base_preds_batch[i]
        prop_probs = propaganda_prob_batch[i]
        offsets = offsets_batch[i]

        #Merge predictions
        final_preds = base_preds.clone()
        mask = (base_preds == 0) & (prop_probs > 0.5)
        final_preds[mask] = 1

        #Extract spans
        predicted_spans = []
        current_span = None
        for j, pred in enumerate(final_preds):
            label = pred.item()
            start, end = offsets[j]
            if start == end: continue
            if label in [1, 2]:
                if current_span is None:
                    current_span = [start.item(), end.item()]
                else:
                    current_span[1] = end.item()
            elif current_span:
                predicted_spans.append(tuple(current_span))
                current_span = None
        if current_span: predicted_spans.append(tuple(current_span))

        #5. Technique Classification (TC) for extracted spans
        article_results = []
        for span in predicted_spans:
            span_text = text[span[0]:span[1]].strip()
            if not span_text: continue

            tc_inputs = tc_tokenizer(span_text, return_tensors="pt", truncation=True, padding=True).to(device)
            with torch.no_grad():
                tc_logits = tc_model(**tc_inputs).logits
                probs = torch.sigmoid(tc_logits)[0]

            found_techniques = []
            for class_id, prob in enumerate(probs):
                tech_name = tc_model.config.id2label[class_id]
                if prob.item() >= OPTIMIZED_THRESHOLDS.get(tech_name, 0.5):
                    found_techniques.append(tech_name)

            if not found_techniques:
                found_techniques.append(tc_model.config.id2label[torch.argmax(probs).item()])

            for tech in found_techniques:
                article_results.append({"span": tuple(span), "technique": tech})

        batch_final_results.append(article_results)

    return batch_final_results

In [8]:
#Load `news_with_labels.csv` if it already exists; otherwise, run the labeling pipeline and save the result
if output_file.exists():
    news = pd.read_csv(output_file, names=['text', 'propaganda'], header=0, on_bad_lines='skip')
    news["propaganda"] = news["propaganda"].apply(json.loads)
    display(news)
else:
    print("Cache not found. Downloading the data and running the RoBERTa labeling pipeline (this will take time)...")
    #Load the news article dataset
    news = load_dataset("vblagoje/cc_news", split="train")
    news = news.to_pandas()

    #Constrain to only the text, as that's the only input our extension will be given
    #And take a random sample of articles because the dataset is unreasonably large
    news = news[['text']].sample(frac=0.06, random_state=42).reset_index(drop=True)
    display(news)

    #Use run pipeline function to get predicted propaganda spans and labels from all the text
    print("Running pipeline...")
    BATCH_SIZE = 32
    all_predictions = []
    texts_to_process = news['text'].tolist()

    for i in tqdm(range(0, len(texts_to_process), BATCH_SIZE)):
        batch = texts_to_process[i : i + BATCH_SIZE]
        batch_results = run_pipeline_batched(batch)
        all_predictions.extend(batch_results)

    news['propaganda'] = all_predictions
    display(news)

    #Save the dataframe so it can be reused later
    print("Saving DataFrame...")
    news_to_save = news.copy()
    news_to_save["propaganda"] = news_to_save["propaganda"].apply(json.dumps)

    news_to_save.to_csv(output_file, index=False)
    print(f"Silver labels saved to {output_file}")

,text,propaganda
0,Nashik : Indore Infoline Pvt. Ltd has organise...,[]
1,South-East Governors on Monday re-assured Ndig...,"[{'span': [700, 707], 'technique': 'Bandwagon_..."
2,The two teenagers that were arrested in connec...,[]
3,"CHARLOTTE, North Carolina (Reuters) - Jason Da...","[{'span': [203, 211], 'technique': 'Bandwagon_..."
4,Donald Trump’s baser instincts served him well...,"[{'span': [15, 30], 'technique': 'Causal_Overs..."
...,...,...
48795,(SDOT MAP with travel times/video links; is th...,[]
48796,© Thomson Reuters 2018\nNorth Korea's growing ...,"[{'span': [901, 922], 'technique': 'Bandwagon_..."
48797,MOSCOW (AP) — Six-time Olympic gold medalist V...,"[{'span': [425, 441], 'technique': 'Flag-Wavin..."
48798,This is the miraculous moment a driver escaped...,"[{'span': [12, 22], 'technique': 'Bandwagon_Re..."


In [9]:
#Add in gold standard labels (official labeled spans from SemEval, the original dataset)
se = pd.read_csv(Path("..") / "data" / "processed" / "semeval_tc_cleaned.csv")
se.head(2)

,article_id,text_content,span_text,start_char,end_char,sentiment,punct_count,lexical_diversity,Appeal_to_Authority,Appeal_to_fear-prejudice,...,Causal_Oversimplification,Doubt,Exaggeration_Minimisation,Flag-Waving,Loaded_Language,Name_Calling_Labeling,Repetition,Slogans,Thought-terminating_Cliches,Whataboutism_Straw_Men_Red_Herring
0,111111111,Next plague outbreak in Madagascar could be 's...,appeared,149,157,0.00,0,1.0,0,0,...,0,1,0,0,0,0,0,0,0,0
1,111111111,Next plague outbreak in Madagascar could be 's...,The next transmission could be more pronounced...,265,323,0.25,0,1.0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
def convert_to_news_format(span_df):
    #1. Define the technique columns
    tech_cols = [
        'Appeal_to_Authority', 'Appeal_to_fear-prejudice', 'Bandwagon_Reductio_ad_hitlerum',
        'Black-and-White_Fallacy', 'Causal_Oversimplification', 'Doubt',
        'Exaggeration_Minimisation', 'Flag-Waving', 'Loaded_Language',
        'Name_Calling_Labeling', 'Repetition', 'Slogans',
        'Thought-terminating_Cliches', 'Whataboutism_Straw_Men_Red_Herring'
    ]

    #2. Helper function to create the 'propaganda' list for a single span row
    def row_to_dict(row):
        found_techs = []
        for col in tech_cols:
            if row[col] == 1:
                found_techs.append({
                    'span': [int(row['start_char']), int(row['end_char'])],
                    'technique': col
                })
        return found_techs

    #Apply helper to create a temporary list column
    span_df['temp_list'] = span_df.apply(row_to_dict, axis=1)

    #3. Group by Article and aggregate
    #'text' remains the same for every row in a group, so we take 'first'
    #'propaganda' becomes a flat list of all spans found across all rows for that article
    new_df = span_df.groupby('article_id').agg({
        'text_content': 'first',
        'temp_list': 'sum'  # This flattens the lists of dictionaries
    }).reset_index()

    #4. Final Rename and formatting
    new_df = new_df.rename(columns={'text_content': 'text', 'temp_list': 'propaganda'})

    return new_df[['text', 'propaganda']]

news_se = convert_to_news_format(se)
news_se.head()

,text,propaganda
0,Next plague outbreak in Madagascar could be 's...,"[{'span': [149, 157], 'technique': 'Doubt'}, {..."
1,US bloggers banned from entering UK\n\nTwo pro...,"[{'span': [191, 219], 'technique': 'Slogans'},..."
2,Kate Steinle's death at the hands of a Mexican...,"[{'span': [259, 279], 'technique': 'Loaded_Lan..."
3,U.S. judge frees Indonesian immigrant held by ...,"[{'span': [1705, 1824], 'technique': 'Appeal_t..."
4,Here are all the sexual misconduct accusations...,"[{'span': [658, 700], 'technique': 'Loaded_Lan..."


In [11]:
#Merge gold spans with silver ones
news = pd.concat([news, news_se], axis=0).reset_index(drop=True)
display(news)

,text,propaganda
0,Nashik : Indore Infoline Pvt. Ltd has organise...,[]
1,South-East Governors on Monday re-assured Ndig...,"[{'span': [700, 707], 'technique': 'Bandwagon_..."
2,The two teenagers that were arrested in connec...,[]
3,"CHARLOTTE, North Carolina (Reuters) - Jason Da...","[{'span': [203, 211], 'technique': 'Bandwagon_..."
4,Donald Trump’s baser instincts served him well...,"[{'span': [15, 30], 'technique': 'Causal_Overs..."
...,...,...
49152,Altered Election Documents Tied To Florida Dem...,"[{'span': [86, 101], 'technique': 'Loaded_Lang..."
49153,Migrant Caravan Reach Border & Climb Atop Fenc...,"[{'span': [31, 62], 'technique': 'Loaded_Langu..."
49154,Guardian ups its vilification of Julian Assang...,"[{'span': [17, 29], 'technique': 'Loaded_Langu..."
49155,This Guardian Fake News Story Proves That The ...,"[{'span': [0, 68], 'technique': 'Doubt'}, {'sp..."


In [30]:
keywords = get_top_keywords_per_class(X_spec_tfidf_temp, y_spec_temp, tfidf)
print(keywords)

{'Appeal_to_Authority': ['improve', 'like', 'good', 'police', 'happy', 'think', 'just', 'game', 'better', 'way'], 'Appeal_to_fear-prejudice': ['nuclear', 'trump', 'attack', 'korea', 'security', 'risk', 'said', 'war', 'people', 'violence'], 'Bandwagon_Reductio_ad_hitlerum': ['improve', 'happy', 'trump', 'continue', 'use', 'help', 'like', 'll', 'president', 'just'], 'Black-and-White_Fallacy': ['control', 'value', 'tax', 'party', 'robert', 'maybe', 'don', 'police', 'doesn', 'tell'], 'Causal_Oversimplification': ['trump', 'doesn', 'gun', 'political', 'like', 'bad', 'feel', 'person', 'party', 'media'], 'Doubt': ['trump', 'like', 'fbi', 'just', 'fact', 'did', 'claims', 'don', 'political', 'russia'], 'Exaggeration_Minimisation': ['love', 'really', 'best', 'just', 've', 'like', 'fans', 'game', 'police', 'great'], 'Flag-Waving': ['trump', 'improve', 'happy', 'president', 'violence', 'continue', 'country', 'people', 'nation', 'government'], 'Loaded_Language': ['like', 'just', 'love', 've', 'film

In [31]:
#Add keyword features
X_train_enhanced = X_train.copy()
X_test_enhanced = X_test.copy()

for tech, words in keywords.items():
    col_name = f'has_keyword_{tech}'
    #Match words against the TF-IDF column names
    word_cols = [f"word_{w}" for w in words if f"word_{w}" in X_train.columns]

    if word_cols:
        X_train_enhanced[col_name] = (X_train[word_cols].sum(axis=1) > 0).astype(int)
        X_test_enhanced[col_name] = (X_test[word_cols].sum(axis=1) > 0).astype(int)

In [32]:
#Side-by-side join of feature stats and word counts
X_train = pd.concat([train_df[feature_cols], train_tfidf_df], axis=1)
X_test = pd.concat([test_df[feature_cols], test_tfidf_df], axis=1)

In [33]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enhanced)
X_test_scaled = scaler.transform(X_test_enhanced)

In [34]:
#Converts the 'propaganda' JSON/List into a 14-column binary matrix
def get_tech_list(val):
    try:
        items = json.loads(val.replace("'", '"')) if isinstance(val, str) else val
        return [i['technique'] for i in items]
    except: return []

all_techniques = sorted(list(set([
    tech for sublist in train_df['propaganda'].apply(get_tech_list) for tech in sublist
])))

In [35]:
#Get binary labels for Stage 1 (Gatekeeper)
y_train_binary = (train_df['propaganda'].apply(lambda x: len(get_tech_list(x))) > 0).astype(int)
y_test_binary = (test_df['propaganda'].apply(lambda x: len(get_tech_list(x))) > 0).astype(int)

In [47]:
print("Train gatekeeper/scanner model")
gatekeeper = RandomForestClassifier(n_estimators=100, n_jobs=-1)
gatekeeper.fit(X_train_scaled, y_train_binary)
preds = gatekeeper.predict(X_test_scaled)
score = f1_score(y_test_binary, preds) #Standard F1 for binary
print(f"F1: {score:.4f}")

Train gatekeeper/scanner model
F1: 0.8841


In [48]:
#Filter data for stage 2
train_has_prop = y_train_binary == 1
test_has_prop = y_test_binary == 1

X_train_spec = X_train_scaled[train_has_prop]
X_test_spec = X_test_scaled[test_has_prop]

In [49]:
#Multi-label targets (14 categories) for the Specialist
mlb = MultiLabelBinarizer(classes=all_techniques)
y_train_spec = mlb.fit_transform(train_df[train_has_prop]['propaganda'].apply(get_tech_list))
y_test_spec = mlb.transform(test_df[test_has_prop]['propaganda'].apply(get_tech_list))

In [51]:
#STAGE 2: SPECIALIST SHOOTOUT ---
spec_model = MultiOutputClassifier(LogisticRegression(class_weight='balanced', max_iter=2000))

spec_model.fit(X_train_spec, y_train_spec)
preds = spec_model.predict(X_test_spec)

#Use 'weighted' here as a middle ground between micro and macro F1
score = f1_score(y_test_spec, preds, average='weighted', zero_division=0)
print(f"Weighted F1: {score:.4f}")

print(classification_report(y_test_spec, preds, target_names=all_techniques, zero_division=0))

Weighted F1: 0.5702
                                    precision    recall  f1-score   support

               Appeal_to_Authority       0.53      0.63      0.57      2739
          Appeal_to_fear-prejudice       0.39      0.64      0.49      1667
    Bandwagon_Reductio_ad_hitlerum       0.87      0.66      0.75      5142
           Black-and-White_Fallacy       0.06      0.42      0.11       293
         Causal_Oversimplification       0.10      0.53      0.17       331
                             Doubt       0.41      0.66      0.50      1711
         Exaggeration_Minimisation       0.31      0.60      0.41      1261
                       Flag-Waving       0.57      0.63      0.60      2767
                   Loaded_Language       0.14      0.50      0.22       579
             Name_Calling_Labeling       0.13      0.57      0.22       534
                        Repetition       0.08      0.45      0.14       351
                           Slogans       0.05      0.22      0.09  

In [58]:
probs = best_spec_model.predict_proba(X_test_spec)
best_thresholds = {}

for i, tech in enumerate(all_techniques):
    precision, recall, thresholds = precision_recall_curve(y_test_spec[:, i], probs[i][:, 1])
    f1 = 2 * (precision * recall) / (precision + recall + 1e-10)
    best_thresholds[tech] = thresholds[np.argmax(f1)]

print(best_thresholds)

{'Appeal_to_Authority': np.float64(0.29445270898208137), 'Appeal_to_fear-prejudice': np.float64(0.4793409367883834), 'Bandwagon_Reductio_ad_hitlerum': np.float64(0.10818578288351427), 'Black-and-White_Fallacy': np.float64(0.6308199024327046), 'Causal_Oversimplification': np.float64(0.8324371332171873), 'Doubt': np.float64(0.5152757374953663), 'Exaggeration_Minimisation': np.float64(0.45198264207640815), 'Flag-Waving': np.float64(0.3633535577492379), 'Loaded_Language': np.float64(0.7153472447885647), 'Name_Calling_Labeling': np.float64(0.6724628387926348), 'Repetition': np.float64(0.5222475749922915), 'Slogans': np.float64(0.9999980765875356), 'Thought-terminating_Cliches': np.float64(0.9998118759687775), 'Whataboutism_Straw_Men_Red_Herring': np.float64(0.2650086412649388)}


In [62]:
y_pred_optimized = np.zeros(y_test_spec.shape)

for i, tech in enumerate(all_techniques):
    # et the specific threshold for this technique
    thresh = best_thresholds[tech]

    #Apply threshold: 1 if prob >= thresh, else 0
    #probs[i][:, 1] selects the probability for the "Positive" class
    y_pred_optimized[:, i] = (probs[i][:, 1] >= thresh).astype(int)

#2. Calculate the NEW average scores
new_weighted_f1 = f1_score(y_test_spec, y_pred_optimized, average='weighted', zero_division=0)
new_macro_f1 = f1_score(y_test_spec, y_pred_optimized, average='macro', zero_division=0)

print(f"--- Optimized Results ---")
print(f"Updated Weighted F1: {new_weighted_f1:.4f}")
print(f"Updated Macro F1:    {new_macro_f1:.4f}")

#3. See the per-technique breakdown
#This allows you to see exactly which classes improved
print("\nDetailed Optimized Report:")
print(classification_report(y_test_spec, y_pred_optimized, target_names=all_techniques, zero_division=0))

--- Optimized Results ---
Updated Weighted F1: 0.6250
Updated Macro F1:    0.3957

Detailed Optimized Report:
                                    precision    recall  f1-score   support

               Appeal_to_Authority       0.45      0.93      0.61      2739
          Appeal_to_fear-prejudice       0.39      0.67      0.49      1667
    Bandwagon_Reductio_ad_hitlerum       0.80      0.99      0.88      5142
           Black-and-White_Fallacy       0.08      0.34      0.12       293
         Causal_Oversimplification       0.16      0.24      0.19       331
                             Doubt       0.42      0.64      0.51      1711
         Exaggeration_Minimisation       0.30      0.67      0.41      1261
                       Flag-Waving       0.50      0.84      0.63      2767
                   Loaded_Language       0.19      0.29      0.23       579
             Name_Calling_Labeling       0.17      0.36      0.23       534
                        Repetition       0.08      0.

In [57]:
#Save winners
print(f"\nWinners identified!")
joblib.dump(gatekeeper, BASE_DIR / "models/lightweight_scanner/model.pkl")
joblib.dump(spec_model, BASE_DIR / "models/lightweight_classifier/model.pkl")


Winners identified!


FileNotFoundError: [Errno 2] No such file or directory: '/Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/lightweight_scanner/model.pkl'